# FILIP GUARD — validare AI cloud, de pe telefon

Acesta este un test interactiv pe exemple sintetice, nu găzduirea aplicației.
Folosește **runtime CPU gratuit**, fără GPU, Colab Pro sau unități plătite.

1. Creează o cheie în organizația **Free** din https://console.groq.com/keys . Nu adăuga card și nu activa Developer.
2. În bara laterală Colab, deschide **Secrets (cheia)**, adaugă `GROQ_API_KEY` și permite accesul acestui notebook. Nu scrie cheia în cod, GitHub sau chat.
3. Confirmă cele trei căsuțe de mai jos și rulează celulele. Testul face cel mult șapte apeluri, la minimum 61 de secunde distanță. Lipsa cotelor produce fallback, nu retry sau upgrade.

Datele trimise sunt exclusiv exemple publice sintetice (engleză/română), fără acces la WordPress, parole sau date ale clienților. Rezultatul nu declară automat calitatea validată. Verifică exemplele de recomandări și descarcă raportul fără secrete.


In [ ]:
import os, pathlib, subprocess, sys
os.umask(0o077)
checkout=pathlib.Path('/content/filip-guard-ai-validation')
if not checkout.exists():
    subprocess.run(['git','clone','--depth','1','--single-branch','--branch','filip-guard-development',
        'https://github.com/Ciobanu87/filip-guard.git',str(checkout)],check=True,capture_output=True)
revision=subprocess.check_output(['git','rev-parse','HEAD'],cwd=checkout,text=True).strip()
print('Repository development revision:',revision)
subprocess.run([sys.executable,'-m','pip','install','-q','beautifulsoup4>=4.12,<5','cryptography>=44,<49','requests>=2.32,<3'],check=True)
sys.path.insert(0,str(checkout))


In [ ]:
#@title Confirmări obligatorii — nu activa plăți
free_organization_without_billing = False #@param {type:"boolean"}
synthetic_data_transfer_authorized = False #@param {type:"boolean"}
seven_live_requests_authorized = False #@param {type:"boolean"}
if not all((free_organization_without_billing,synthetic_data_transfer_authorized,seven_live_requests_authorized)):
    raise RuntimeError('Confirmă organizația Free fără facturare și testele sintetice înainte de rulare.')
from google.colab import userdata
from advisor_groq import GroqFreeProvider
from contextual_advisor import ContextualAdvisor
from validate_cloud_advisor import run_validation
import re
try:
    _key_check=userdata.get('GROQ_API_KEY')
    if not isinstance(_key_check,str) or not re.fullmatch(r'gsk_[A-Za-z0-9_-]{16,200}',_key_check):raise ValueError()
except Exception:
    raise RuntimeError('Adaugă GROQ_API_KEY în Colab Secrets și permite accesul notebookului. Nu lipi cheia în cod.') from None
finally:
    _key_check=None
provider=GroqFreeProvider(lambda:userdata.get('GROQ_API_KEY'),free_plan_confirmed=True,data_consent=True)
advisor=ContextualAdvisor(checkout/'validation-ledger.sqlite',provider,daily_calls=7,timeout=20)
print('Configurație pregătită. Nicio inferență nu a fost executată de această celulă.')


In [ ]:
import json
report=run_validation(advisor,live=True,progress=lambda result:print(json.dumps(result,ensure_ascii=False),flush=True))
report['repository_revision']=revision
print(json.dumps(report,ensure_ascii=False,indent=2))
print('Inferență reală nouă:',report['real_inference_executed'])
print('Calitatea necesită evaluare umană:',not report['quality_validated'])


In [ ]:
import uuid
from google.colab import files
report_path=checkout/('cloud-validation-'+uuid.uuid4().hex+'.json')
with report_path.open('x',encoding='utf-8') as out:
    json.dump(report,out,ensure_ascii=False,indent=2)
report_path.chmod(0o600)
files.download(str(report_path))
